In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dev = pd.read_csv("../data/casas_dev.csv")

1.1

Fragmento random del dataset

In [ ]:
dev.sample(30)

- Columna unidades: deberia estar en una misma medida, elijo m2
- Columna pisos: demasiados NaN
- Columna pileta: convertir en 1(True) y 0(False)
- Columna tipo: pasar de variable categorica a numero
- Normalizar las variables (las escalas son demasiado distintas)

Busco NaNs en cada columna para ver si elimino alguna:

In [ ]:
columnas = []
porcentajes = []
for col in dev.columns:
	columna = dev[col]
	total_NaNs_en_columna = columna.isna().sum() #sumo cantidad de NaNs 
	porcentaje_NaNs_en_columna = (total_NaNs_en_columna/dev.shape[0]) * 100

	columnas.append(col)
	porcentajes.append(porcentaje_NaNs_en_columna)

plt.figure()
plt.bar(columnas, porcentajes)
plt.xticks(rotation=45)
plt.ylabel("Porcentaje de NaNs")
plt.title("Porcentaje de valores faltantes por columna")
plt.tight_layout()
plt.show()


Decido eliminar la columna de NaNs porque hay mas de un 50% de valores faltantes, lo que puede producir sesgos importantes. 

Las demas no tienen  porcentajes tan significativos 
--> utilizo mediana para estimar valores numericos


Analizo outliers 

In [ ]:
for col in dev.select_dtypes(include=['number']).columns: #recorre solo columnas numericas
	columna = dev[col].dropna()
	plt.figure()
	plt.boxplot(columna)
	plt.title(f'Boxplot de {col}')
	plt.show()

Analisis Boxplots (tomando en cuenta lo importante): 

- Precios: 
Mayoria de las propiedades con precios bajos, pero hay valores extremadamente altos (outliers) que generan sesgo hacia valores mayores.

- Area:
Mayoria de las propiedades con terrenos chicos, pero hay terrenos extremadamente altos que generan sesgo hacia valores altos.

- Metros Cubiertos:
Valores concentrados en rangos bajos. No hay outliers importantes pero si hay sesgo hacia valores altos.

- Ambientes:
Tiene dispersion. Hay sesgo (leve) de valores chicos.

- Pisos:
Tiene dispersion. Hay sesgo (leve) de valores chicos.

- Latitud y longitud:
Ambas tienen demasiada dispersion.

- Edad:
Valores concentrados abajo (casas relativamente nuevas), otliers muy altos, generando sesgo hacia valores altos.



Soluciones:
Alta dispersion: Con normalizacion se lleva a todas las variables a la misma escala mejorando el entrenamiento del modelo

Sesgos grandes:
Se tienen en cuenta en el modelo
Solucion: aplicar transformacionesm 

1.2

In [ ]:
for col in dev.select_dtypes(include=['number']).columns:
    if col != 'precio':
        sns.scatterplot(x=dev[col], y=dev['precio'])
        plt.title(f'precio vs {col}')
        plt.show()

Relaciones importantes:
- Precio vs Area/Metro cubiertos:
Cuanto mas terreno --> mas alto el precio

- Precio vs ambientes:
Con mas ambientes aumenta el precio pero hay gran dispersion

Otras:
- Precio vs edad:
Tendencia poco clara y mucho ruido
- Latitud y longitud:
No se puede encontrar ninguna relacion


Con las relaciones importantes podremos entrenar a nuestro modelo para que logre predir correctamente. Las otras no aportan mucho para nuestro modelo (por el momento).

1.3

Separo dataset (80% train - 20% validation)

In [43]:
dev = dev.sample(frac=1, random_state=42) 

Uso random_state para que siempre se mezcle igual y comparar resultados

In [45]:
total_filas = dev.shape[0]
q_filas_train = int(0.8*total_filas)
train_set = dev[:q_filas_train]
validation_set = dev[q_filas_train:]

train_set.to_csv("../data/casas_train.csv", index=False)
validation_set.to_csv("../data/casas_validation.csv", index=False)